## Cloning the repository - Google Colab

This step is needed for importing the src scripts. This is needed when execution through google colab

In [1]:
!pip install -U ipython

^C


In [ ]:
%load_ext autoreload
%autoreload 2

#Clone the repository
import os
if not os.path.exists("your-project"):
    !git clone https://github.com/ras112git/predictive_modeling_and_mobility_forecasting.git
else:
    print("Repo already cloned.")


%cd predictive_modeling_and_mobility_forecasting

# Install dependencies
!pip install -r requirements.txt

import sys
sys.path.append(os.getcwd())

## Setting up in the virtual environment

This cell need to be run, when the data will be cleaned localy.

In [2]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Make src/ importable: add the project root (parent of notebooks/) to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Run from the project root so relative paths like "data/raw/..." resolve
os.chdir(project_root)
print(f"Working dir: {os.getcwd()}")

Working dir: c:\Users\rcsrc\Documents\GitHub\predictive_modeling_and_mobility_forecasting


## Data cleaning process

This section summarizes all the data cleaning steps taken in the dataset, before starting with the training.

In [3]:
from src.download_data import download_raw_data

train_path, test_path = download_raw_data()

# Import the raw datasets
import pandas as pd
import numpy as np

df_train = pd.read_csv(train_path)
# df_test = pd.read_csv(test_path)

# and also saving the data IDs, as later will be used 
# df_train['datetime'] = pd.to_datetime(df_train['datetime'], utc=True).dt.tz_localize(None)

#Transform the datetime, needed for the submission file, the localize makes sure that the format is correct (rather than having the UCT reference)
df_train['datetime'] = pd.to_datetime(df_train['datetime'], utc=True).dt.tz_localize(None)

# The insert method takes (position_index, column_name, values)
df_train.insert(0, 'id', df_train.iloc[:, 0].astype(str) + "_" + df_train.iloc[:, 1].astype(str))


Raw data already exists at data/raw\dataset_train.csv, skipping download.
Raw data already exists at data/raw\dataset_test.csv, skipping download.


In [4]:
df_train

,id,datetime,station_number,name,lat,lng,bikes
0,2024-09-03 17:30:00_32000,2024-09-03 17:30:00,32000,Julius-Raab-Platz,48.211544,16.382374,25
1,2024-09-03 17:30:00_32001,2024-09-03 17:30:00,32001,Hoher Markt,48.210666,16.372983,14
2,2024-09-03 17:30:00_32002,2024-09-03 17:30:00,32002,Oper,48.202683,16.369702,9
3,2024-09-03 17:30:00_32003,2024-09-03 17:30:00,32003,Volksgarten,48.206516,16.360400,3
4,2024-09-03 17:30:00_32004,2024-09-03 17:30:00,32004,Taborstraße U2,48.219522,16.382218,5
...,...,...,...,...,...,...,...
1824915,2025-03-06 10:00:00_32273,2025-03-06 10:00:00,32273,Modecenterstraße / The Marks,48.184348,16.413837,4
1824916,2025-03-06 10:00:00_32274,2025-03-06 10:00:00,32274,Am Langen Felde,48.250336,16.450206,5
1824917,2025-03-06 10:00:00_32277,2025-03-06 10:00:00,32277,eLastenräder - Bruno-Marek-Allee 6,48.227914,16.391516,1
1824918,2025-03-06 10:00:00_32278,2025-03-06 10:00:00,32278,eLastenräder - Am Tabor 23,48.224598,16.392090,2


### Data cleaning.
First we need to remove all the Na factors in our dataset. 

In [ ]:
# Not implemented yet: all the data cleaning procedures to go directly in the source
# from src.data_cleaning import clean_data

# View the actual NA rows
df_train[df_train.isna().any(axis=1)]

# If there are rows with NA, then drop them
if df_train.isna().any(axis=1).any():
    df_train = df_train.dropna()


#No rows are NA

ImportError: cannot import name 'clean_data' from 'src.data_cleaning' (c:\Users\rcsrc\Documents\GitHub\predictive_modeling_and_mobility_forecasting\src\data_cleaning.py)

In [ ]:
#Find duplicates (and show)
df_train[df_train.duplicated()]

# If there are duplicates, then drop them.
if df_train.duplicated().any():
    df_train = df_train.drop_duplicates()

## Feature engineering

Different processes to improve the features given. Mainly
- Transform into pd.datetime (done at the data import procedure)
- Separate year, month, days, hours and minutes
- Create cyclical variables (for month and year)
- Add if this day was specifically a holiday
- (Not yet implemented) A sort of lag variable (how many bikes were there in the previous step), quite important
- (Not yet implemented) Calculate the neighbourhood, how bussy the neighbourhood is 

In [4]:
#I want to split my datainto the datapart (how already fastai tabular uses to learn)
df_train['hour'] = df_train['datetime'].dt.hour
df_train['minute'] = df_train['datetime'].dt.minute  # 0 or 30
df_train['dayofweek'] = df_train['datetime'].dt.dayofweek
df_train['month'] = df_train['datetime'].dt.month
df_train['is_weekend'] = df_train['dayofweek'].isin([5, 6]).astype(int)

# For cyclical data I can help the model differenciate between simple ordered rankings and actual circular data. In this case I do it with the hour (so it does not jump from 23 to 0 directly)
df_train['hour_sin'] = np.sin(2 * np.pi * df_train['hour'] / 24)
df_train['hour_cos'] = np.cos(2 * np.pi * df_train['hour'] / 24)


In [7]:
df_train

,id,datetime,station_number,name,lat,lng,bikes,hour,minute,dayofweek,month,is_weekend,hour_sin,hour_cos
0,2024-09-03 17:30:00_32000,2024-09-03 17:30:00,32000,Julius-Raab-Platz,48.211544,16.382374,25,17,30,1,9,0,-0.965926,-0.258819
1,2024-09-03 17:30:00_32001,2024-09-03 17:30:00,32001,Hoher Markt,48.210666,16.372983,14,17,30,1,9,0,-0.965926,-0.258819
2,2024-09-03 17:30:00_32002,2024-09-03 17:30:00,32002,Oper,48.202683,16.369702,9,17,30,1,9,0,-0.965926,-0.258819
3,2024-09-03 17:30:00_32003,2024-09-03 17:30:00,32003,Volksgarten,48.206516,16.360400,3,17,30,1,9,0,-0.965926,-0.258819
4,2024-09-03 17:30:00_32004,2024-09-03 17:30:00,32004,Taborstraße U2,48.219522,16.382218,5,17,30,1,9,0,-0.965926,-0.258819
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1824915,2025-03-06 10:00:00_32273,2025-03-06 10:00:00,32273,Modecenterstraße / The Marks,48.184348,16.413837,4,10,0,3,3,0,0.500000,-0.866025
1824916,2025-03-06 10:00:00_32274,2025-03-06 10:00:00,32274,Am Langen Felde,48.250336,16.450206,5,10,0,3,3,0,0.500000,-0.866025
1824917,2025-03-06 10:00:00_32277,2025-03-06 10:00:00,32277,eLastenräder - Bruno-Marek-Allee 6,48.227914,16.391516,1,10,0,3,3,0,0.500000,-0.866025
1824918,2025-03-06 10:00:00_32278,2025-03-06 10:00:00,32278,eLastenräder - Am Tabor 23,48.224598,16.392090,2,10,0,3,3,0,0.500000,-0.866025


In [5]:
# Add a row that tells me if that day was a holiday (maybe remove later on)
import holidays

# Create Austria holidays object
at_holidays = holidays.Austria(subdiv='9') #subdiv = 9 locates the specific holidays of viena


# Create the holiday column
df_train['is_holiday'] = df_train['datetime'].dt.date.apply(lambda x: x in at_holidays)

# Optionally, get the holiday name
# df_train['holiday_name'] = df_train['datetime'].dt.date.apply(lambda x: at_holidays.get(x))


In [9]:
df_train

,id,datetime,station_number,name,lat,lng,bikes,hour,minute,dayofweek,month,is_weekend,hour_sin,hour_cos,is_holiday,holiday_name
0,2024-09-03 17:30:00_32000,2024-09-03 17:30:00,32000,Julius-Raab-Platz,48.211544,16.382374,25,17,30,1,9,0,-0.965926,-0.258819,False,NaN
1,2024-09-03 17:30:00_32001,2024-09-03 17:30:00,32001,Hoher Markt,48.210666,16.372983,14,17,30,1,9,0,-0.965926,-0.258819,False,NaN
2,2024-09-03 17:30:00_32002,2024-09-03 17:30:00,32002,Oper,48.202683,16.369702,9,17,30,1,9,0,-0.965926,-0.258819,False,NaN
3,2024-09-03 17:30:00_32003,2024-09-03 17:30:00,32003,Volksgarten,48.206516,16.360400,3,17,30,1,9,0,-0.965926,-0.258819,False,NaN
4,2024-09-03 17:30:00_32004,2024-09-03 17:30:00,32004,Taborstraße U2,48.219522,16.382218,5,17,30,1,9,0,-0.965926,-0.258819,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1824915,2025-03-06 10:00:00_32273,2025-03-06 10:00:00,32273,Modecenterstraße / The Marks,48.184348,16.413837,4,10,0,3,3,0,0.500000,-0.866025,False,NaN
1824916,2025-03-06 10:00:00_32274,2025-03-06 10:00:00,32274,Am Langen Felde,48.250336,16.450206,5,10,0,3,3,0,0.500000,-0.866025,False,NaN
1824917,2025-03-06 10:00:00_32277,2025-03-06 10:00:00,32277,eLastenräder - Bruno-Marek-Allee 6,48.227914,16.391516,1,10,0,3,3,0,0.500000,-0.866025,False,NaN
1824918,2025-03-06 10:00:00_32278,2025-03-06 10:00:00,32278,eLastenräder - Am Tabor 23,48.224598,16.392090,2,10,0,3,3,0,0.500000,-0.866025,False,NaN


In [6]:
# and now save it to the interim path(maybe we change this later for full path) 
from pathlib import Path

out_path = Path('data/processed/cleaned_data.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df_train.to_csv(out_path, index=False)